# NeoLM — Tokenizer Vocabulary Benchmark

Compare ByteLevel BPE vocabulary sizes on NeoCorpus.

## Experiment

- Training sample: 200,000 documents
- Validation sample: 2,000 documents
- Vocabulary sizes: 8K, 16K, 32,768, 49,152, 50,258
- Random seed: 1337

## Evaluation

- Validation token count
- Characters and bytes per token
- Document token-length distribution
- Embedding parameter cost

The released NeoLM uses a vocabulary of 49,152 tokens.

This benchmark evaluates tokenizer efficiency, not language
model generation quality.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_from_disk
from tokenizers import Tokenizer, Regex, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel, Sequence
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
import gc
import time
import matplotlib.pyplot as plt

In [ ]:
DB_DIR = Path("../../artifacts/pretrain_db/final")

SEED = 1337
TRAIN_SAMPLE_SIZE = 200_000
VAL_SAMPLE_SIZE = 2_000

VOCAB_SIZES = [8_000, 16_000, 32_768, 49_152, 50_258]
BATCH_SIZE = 20_000

db = load_from_disk(str(DB_DIR))

rng = np.random.default_rng(SEED)

train_indices = np.sort(
    rng.choice(len(db["train"]), TRAIN_SAMPLE_SIZE, replace=False)
)

val_indices = np.sort(
    rng.choice(len(db["validation"]), VAL_SAMPLE_SIZE, replace=False)
)

benchmark_train = db["train"].select(train_indices)
benchmark_val = db["validation"].select(val_indices)

print("Train sample:", len(benchmark_train))
print("Validation sample:", len(benchmark_val))
print("Vocabulary sizes:", VOCAB_SIZES)

In [ ]:
SPECIAL_TOKENS = [
    "<|endoftext|>",
    "<|pad|>",
]

PATTERN = (
    r"""'(?i:[sdmt]|ll|ve|re)|"""
    r"""[^\r\n\p{L}\p{N}]?+\p{L}++|"""
    r"""\p{N}{1,3}+|"""
    r""" ?[^\s\p{L}\p{N}]++[\r\n]*+|"""
    r"""\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
)


def build_tokenizer():
    tokenizer = Tokenizer(BPE(unk_token=None))

    tokenizer.pre_tokenizer = Sequence([
        pre_tokenizers.Split(
            Regex(PATTERN),
            behavior="isolated",
            invert=False,
        ),
        ByteLevel(
            add_prefix_space=False,
            use_regex=False,
        ),
    ])

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


def batch_iterator(dataset):
    for start in range(0, len(dataset), BATCH_SIZE):
        yield dataset[start:start + BATCH_SIZE]["text"]

In [ ]:
results = []
token_lengths = {}

validation_texts = benchmark_val["text"]
validation_chars = sum(len(t) for t in validation_texts)
validation_bytes = sum(len(t.encode("utf-8")) for t in validation_texts)

for vocab_size in VOCAB_SIZES:
    tok = build_tokenizer()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=SPECIAL_TOKENS,
        initial_alphabet=ByteLevel.alphabet(),
        show_progress=True,
    )

    # Training
    start = time.perf_counter()

    tok.train_from_iterator(
        batch_iterator(benchmark_train),
        trainer=trainer,
        length=len(benchmark_train),
    )

    training_seconds = time.perf_counter() - start
    actual_vocab = tok.get_vocab_size()

    # Evaluation
    lengths = []

    for i in range(0, len(validation_texts), 256):
        encodings = tok.encode_batch(
            validation_texts[i:i + 256],
            add_special_tokens=False,
        )

        lengths.extend(
            len(enc.ids) + 1 for enc in encodings  # EOS
        )

    lengths = np.asarray(lengths, dtype=np.int64)
    total_tokens = int(lengths.sum())

    # Round-trip validation
    for text in validation_texts[:100]:
        decoded = tok.decode(
            tok.encode(
                text,
                add_special_tokens=False,
            ).ids,
            skip_special_tokens=False,
        )

        assert decoded == text, "Tokenizer round-trip failed"

    # Metrics
    stats = {
        "vocab_size": vocab_size,
        "actual_vocab": actual_vocab,
        "total_tokens": total_tokens,
        "chars_per_token": validation_chars / total_tokens,
        "bytes_per_token": validation_bytes / total_tokens,
        "avg_tokens": float(lengths.mean()),
        "p50": float(np.percentile(lengths, 50)),
        "p95": float(np.percentile(lengths, 95)),
        "p99": float(np.percentile(lengths, 99)),
        "embedding_params_M": actual_vocab * 512 / 1e6,
        "training_seconds": training_seconds,
    }

    results.append(stats)
    token_lengths[vocab_size] = lengths

    # Summary
    print(f"\n{'-' * 65}")
    print(f"RESULTS | VOCAB SIZE: {vocab_size:,}")
    print(f"{'-' * 65}")

    print(
        f"Actual vocab:   {actual_vocab:,}\n"
        f"Total tokens:   {total_tokens:,}\n"
        f"Chars/token:    {stats['chars_per_token']:.3f}\n"
        f"Bytes/token:    {stats['bytes_per_token']:.3f}\n"
        f"Avg tokens/doc: {stats['avg_tokens']:,.1f}\n"
        f"P50 / P95 / P99: "
        f"{stats['p50']:,.0f} / "
        f"{stats['p95']:,.0f} / "
        f"{stats['p99']:,.0f}\n"
        f"Embedding:      {stats['embedding_params_M']:.2f}M params\n"
        f"Training time:  {training_seconds:.1f}s\n"
        f"Round-trip:     PASSED (100 samples)"
    )

    del tok
    gc.collect()

# Final comparison
results_df = pd.DataFrame(results).sort_values("vocab_size")

print(f"\n{'=' * 65}")
print("FINAL TOKENIZER BENCHMARK")
print(f"{'=' * 65}")

display(results_df.round(3))

print("\nALL TOKENIZER CANDIDATES EVALUATED ✅")

In [ ]:
df = results_df.sort_values("vocab_size")

plt.figure(figsize=(9, 5))

plt.plot(
    df["vocab_size"],
    df["total_tokens"] / 1e6,
    marker="o",
)

for _, row in df.iterrows():
    plt.annotate(
        f'{int(row["vocab_size"]):,}',
        (row["vocab_size"], row["total_tokens"] / 1e6),
        xytext=(0, 9),
        textcoords="offset points",
        ha="center",
    )

plt.xlabel("Vocabulary Size")
plt.ylabel("Validation Tokens (Millions)")
plt.title("Vocabulary Size vs Tokenization Efficiency")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    df["embedding_params_M"],
    df["bytes_per_token"],
    marker="o",
)

for _, row in df.iterrows():
    plt.annotate(
        f'{int(row["vocab_size"]):,}',
        (row["embedding_params_M"], row["bytes_per_token"]),
        xytext=(6, -12),
        textcoords="offset points",
    )

plt.xlabel("Embedding Parameters (Millions)")
plt.ylabel("Bytes per Token")
plt.title("Tokenization Efficiency vs Embedding Cost")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Research Conclusion

### Results

Five ByteLevel BPE tokenizers were trained on the same 200,000-document sample and evaluated on 2,000 validation documents.

All candidates passed the encoding/decoding round-trip checks.

### Key Findings

- Increasing vocabulary size consistently reduced token counts.
- Moving from **32,768 to 49,152** reduced validation tokens by approximately **2.50%**, while adding **8.39M embedding parameters**.
- Moving from **49,152 to 50,258** reduced validation tokens by only **0.12%**, suggesting diminishing compression gains.
- The 32,768 vocabulary remains a strong alternative when parameter efficiency is prioritized.

### Selected Vocabulary: 49,152

We selected **49,152 tokens** for the NeoLM-Small-Base release.

This provides better compression than 32K while avoiding further vocabulary expansion for marginal gains.

However, this benchmark does not establish 49,152 as the globally optimal vocabulary size. It evaluates tokenization efficiency and embedding cost, not downstream language-model quality.

### Limitations

- Tokenizers were trained on a 200K-document subset, not the full corpus.
- Evaluation used 2K validation documents.
- Results may change when training on the full dataset.
- Model quality, training throughput, and inference performance were not measured.

### Next Step

Train the final **49,152-token ByteLevel BPE tokenizer** on the full NeoCorpus training split (1,889,952 documents).
